# Olist E-Commerce — Full Exploratory Data Analysis
**Project:** Olist Business Analytics · **Owner:** Nikhil Harins · **Generated:** 2026-08-06

> Feeds `00_Context/PROJECT_CANVAS.md`. Analysis restricted to **delivered orders**; customer identity via `customer_unique_id`. This notebook is the statistical deep-dive behind the business narrative.

## 1. Setup — load all 9 datasets

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
import warnings; warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 60)
pd.set_option('display.width', 200)

RAW = Path('01_Raw_Data'); OUT = Path('06_AI/Outputs/Generated_Insights')
OUT.mkdir(parents=True, exist_ok=True)

def load(n): return pd.read_csv(RAW/n)
def pload(n, cols):
    df = pd.read_csv(RAW/n)
    for c in cols: df[c] = pd.to_datetime(df[c], errors='coerce')
    return df

orders    = pload('olist_orders_dataset.csv', ['order_purchase_timestamp','order_approved_at','order_delivered_carrier_date','order_delivered_customer_date','order_estimated_delivery_date'])
items     = load('olist_order_items_dataset.csv')
payments  = load('olist_order_payments_dataset.csv')
reviews   = load('olist_order_reviews_dataset.csv')
customers = load('olist_customers_dataset.csv')
sellers   = load('olist_sellers_dataset.csv')
products  = load('olist_products_dataset.csv')
trans     = load('product_category_name_translation.csv')

for nm in ['orders','items','payments','reviews','customers','sellers','products']:
    df = eval(nm); print(f'{nm:10s} {df.shape[0]:>9,} rows x {df.shape[1]} cols')

orders        99,441 rows x 8 cols
items        112,650 rows x 7 cols
payments     103,886 rows x 5 cols
reviews       99,224 rows x 7 cols
customers     99,441 rows x 5 cols
sellers        3,095 rows x 4 cols
products      32,951 rows x 9 cols


## 2. Data Quality Snapshot (DAMA-5 dimensions)

In [2]:
for nm in ['orders','items','payments','reviews','customers','sellers','products']:
    df = eval(nm)
    nulls = df.isna().sum(); nulls = nulls[nulls>0]
    dups  = df.duplicated().sum()
    print(f'--- {nm}: {df.shape[0]:,} rows, {df.shape[1]} cols, dups={dups}')
    if len(nulls):
        for col,v in nulls.items():
            print(f'    {col}: {v:,} null ({v/len(df):.1%})')

--- orders: 99,441 rows, 8 cols, dups=0
    order_approved_at: 160 null (0.2%)
    order_delivered_carrier_date: 1,783 null (1.8%)
    order_delivered_customer_date: 2,965 null (3.0%)


--- items: 112,650 rows, 7 cols, dups=0
--- payments: 103,886 rows, 5 cols, dups=0


--- reviews: 99,224 rows, 7 cols, dups=0
    review_comment_title: 87,656 null (88.3%)
    review_comment_message: 58,247 null (58.7%)


--- customers: 99,441 rows, 5 cols, dups=0
--- sellers: 3,095 rows, 4 cols, dups=0
--- products: 32,951 rows, 9 cols, dups=0
    product_category_name: 610 null (1.9%)
    product_name_lenght: 610 null (1.9%)
    product_description_lenght: 610 null (1.9%)
    product_photos_qty: 610 null (1.9%)
    product_weight_g: 2 null (0.0%)
    product_length_cm: 2 null (0.0%)
    product_height_cm: 2 null (0.0%)
    product_width_cm: 2 null (0.0%)


## 3. Core Money & Scale Facts

In [3]:
o = orders[orders['order_status']=='delivered'].copy()
it_agg = items.groupby('order_id').agg(price=('price','sum'), freight=('freight_value','sum'))
o = o.merge(it_agg, on='order_id', how='inner')

rev = o['price'].sum(); aov = rev/len(o)
print(f'GROSS REVENUE (delivered): R$ {rev:,.0f}')
print(f'DELIVERED ORDERS:          {len(o):,.0f}')
print(f'AVERAGE ORDER VALUE (AOV): R$ {aov:,.2f}')
print(f'TOTAL FREIGHT REVENUE:     R$ {o["freight"].sum():,.0f}  ({o["freight"].sum()/rev:.1%} of revenue)')

cc = customers.merge(orders, on='customer_id')[['customer_unique_id','order_id']]
c_orders = cc.groupby('customer_unique_id')['order_id'].nunique()
print(f'\nUNIQUE CUSTOMERS:      {len(c_orders):,.0f}')
print(f'REPEAT CUSTOMER RATE:  {(c_orders>1).mean():.2%}  ({(c_orders>1).sum():,} customers repeat)')
print(f'AVG ORDERS / CUSTOMER: {c_orders.mean():.3f}')
print(f'CUSTOMERS WITH >=3 ORDERS: {(c_orders>=3).sum():,}  ({(c_orders>=3).mean():.2%})')

GROSS REVENUE (delivered): R$ 13,221,498
DELIVERED ORDERS:          96,478
AVERAGE ORDER VALUE (AOV): R$ 137.04
TOTAL FREIGHT REVENUE:     R$ 2,198,276  (16.6% of revenue)



UNIQUE CUSTOMERS:      96,096
REPEAT CUSTOMER RATE:  3.12%  (2,997 customers repeat)
AVG ORDERS / CUSTOMER: 1.035
CUSTOMERS WITH >=3 ORDERS: 252  (0.26%)


## 4. Growth — Monthly Revenue, Volume, AOV

In [4]:
o['m'] = o['order_purchase_timestamp'].dt.to_period('M')
g = o.groupby('m').agg(orders=('order_id','size'), revenue=('price','sum'), aov=('price','mean'))
g['aov'] = g['revenue']/g['orders']
print('FULL MONTHLY SERIES')
print(g.to_string())
print('\n--- Year-over-year (mature months) ---')
print(f'  Jun-17 orders {g.loc[g.index>=pd.Period("2017-06",freq="M")]["orders"].head(1).iloc[0]}  vs  Jun-18 {g.loc[g.index>=pd.Period("2018-06",freq="M")]["orders"].head(1).iloc[0]}')
print(f'  Peak volume month: {g.orders.idxmax()} ({g.orders.max():,} orders)')
print(f'  Peak revenue month: {g.revenue.idxmax()} (R$ {g.revenue.max():,.0f})')
print(f'  Revenue CAGR (first full yr 2017-01..2017-12 vs 2018-01..2018-08):')

FULL MONTHLY SERIES
         orders    revenue         aov
m                                     
2016-09       1     134.97  134.970000
2016-10     265   40325.11  152.170226
2016-12       1      10.90   10.900000
2017-01     750  111798.36  149.064480
2017-02    1653  234223.40  141.695947
2017-03    2546  359198.85  141.083602
2017-04    2303  340669.68  147.924307
2017-05    3546  489338.25  137.997250
2017-06    3135  421923.37  134.584807
2017-07    3872  481604.52  124.381333
2017-08    4193  554699.70  132.291844
2017-09    4150  607399.67  146.361366
2017-10    4478  648247.65  144.762762
2017-11    7289  987765.37  135.514525
2017-12    5513  726033.19  131.694756
2018-01    7069  924645.00  130.802801
2018-02    6555  826437.13  126.077365
2018-03    7003  953356.25  136.135406
2018-04    6798  973534.09  143.208898
2018-05    6749  977544.69  144.842894
2018-06    6099  856077.86  140.363643
2018-07    6159  867953.46  140.924413
2018-08    6351  838576.64  132.038520

--- 

### 4.1 Growth rate by month (momentum)

In [5]:
g2 = g.copy()
g2['mom'] = g2['orders'].pct_change()*100
g2['revenue_mom'] = g2['revenue'].pct_change()*100
print(g2[['orders','revenue','mom','revenue_mom']].tail(12).to_string())
print(f'\nMedian monthly volume growth: {g2.mom.median():.1f}%')
print(f'Median monthly revenue growth: {g2.revenue_mom.median():.1f}%')

         orders    revenue        mom  revenue_mom
m                                                 
2017-09    4150  607399.67  -1.025519     9.500631
2017-10    4478  648247.65   7.903614     6.725058
2017-11    7289  987765.37  62.773560    52.374694
2017-12    5513  726033.19 -24.365482   -26.497404
2018-01    7069  924645.00  28.224197    27.355748
2018-02    6555  826437.13  -7.271184   -10.621143
2018-03    7003  953356.25   6.834477    15.357384
2018-04    6798  973534.09  -2.927317     2.116506
2018-05    6749  977544.69  -0.720800     0.411963
2018-06    6099  856077.86  -9.631056   -12.425706
2018-07    6159  867953.46   0.983768     1.387210
2018-08    6351  838576.64   3.117389    -3.384608

Median monthly volume growth: 5.0%
Median monthly revenue growth: 8.1%


### 4.2 AOV trend

In [6]:
print(g[['aov']].tail(12).to_string())
print(f'\nAOV 2017 average: R$ {g[g.index.year==2017].aov.mean():,.2f}')
print(f'AOV 2018 average: R$ {g[g.index.year==2018].aov.mean():,.2f}')

                aov
m                  
2017-09  146.361366
2017-10  144.762762
2017-11  135.514525
2017-12  131.694756
2018-01  130.802801
2018-02  126.077365
2018-03  136.135406
2018-04  143.208898
2018-05  144.842894
2018-06  140.363643
2018-07  140.924413
2018-08  132.038520

AOV 2017 average: R$ 138.95
AOV 2018 average: R$ 136.80


## 5. Product Categories — revenue & satisfaction by category

In [7]:
pi = items.merge(products[['product_id','product_category_name']], on='product_id', how='left')
pi = pi.merge(trans, on='product_category_name', how='left')
# keep untranslated PT names (e.g. pc_gamer), only truly missing -> uncategorized
pi['cat'] = pi['product_category_name_english'].fillna(pi['product_category_name']).fillna('uncategorized')
pi['price'] = pi['price'].fillna(0)

cat = pi.groupby('cat').agg(orders=('order_id','nunique'), revenue=('price','sum'), items=('order_item_id','size'), avg_price=('price','mean'))
cat = cat.sort_values('revenue', ascending=False)
print('TOP 15 CATEGORIES BY REVENUE')
print(cat.head(15).to_string())
print(f'\nTop 5 categories = {cat.head(5).revenue.sum()/cat.revenue.sum():.1%} of all category revenue')
uncat_rev = cat.revenue.get('uncategorized', 0.0)
print(f'Total categories: {len(cat)}; uncategorized revenue: R$ {uncat_rev:,.0f}')

TOP 15 CATEGORIES BY REVENUE
                       orders     revenue  items   avg_price
cat                                                         
health_beauty            8836  1258681.34   9670  130.163531
watches_gifts            5624  1205005.68   5991  201.135984
bed_bath_table           9417  1036988.68  11115   93.296327
sports_leisure           7720   988048.97   8641  114.344285
computers_accessories    6689   911954.32   7827  116.513903
furniture_decor          6449   729762.49   8334   87.564494
cool_stuff               3632   635290.85   3796  167.357969
housewares               5884   632248.66   6964   90.788148
auto                     3897   592720.11   4235  139.957523
garden_tools             3518   485256.46   4347  111.630196
toys                     3886   483946.60   4117  117.548360
baby                     2885   411764.89   3065  134.344173
perfumery                3162   399124.87   3419  116.737312
telephony                4199   323667.53   4545   71.21

## 6. Payments — method mix, installments, credit behaviour

In [8]:
pa = payments[payments['order_id'].isin(o['order_id'])]
pt = pa.groupby('payment_type').agg(orders=('order_id','nunique'), value=('payment_value','sum'))
pt['share'] = pt['value']/pt['value'].sum()
print('PAYMENT MIX')
print(pt.to_string())
print(f'\nMean installments: {pa.payment_installments.mean():.2f}')
print(f'Share of revenue in installments>1: {(pa[pa.payment_installments>1].payment_value.sum()/pa.payment_value.sum()):.1%}')
print('\nInstallment distribution (top 8):')
print(pa.payment_installments.value_counts().head(8).to_string())

PAYMENT MIX
              orders        value     share
payment_type                               
boleto         19191   2769932.58  0.179604
credit_card    74304  12101094.88  0.784641
debit_card      1485    208421.12  0.013514
voucher         3679    343013.19  0.022241

Mean installments: 2.85
Share of revenue in installments>1: 63.1%

Installment distribution (top 8):
payment_installments
1     50929
2     12075
3     10164
4      6891
10     5150
5      5095
8      4136
6      3804


## 7. Geography — state concentration

In [9]:
oc = orders[['order_id','customer_id']].merge(customers[['customer_id','customer_state']], on='customer_id')
oc = oc.merge(items[['order_id','price']], on='order_id', how='inner')
st = oc.groupby('customer_state').agg(orders=('order_id','nunique'), revenue=('price','sum'))
st = st.sort_values('revenue', ascending=False)
print('STATE REVENUE RANKING')
print(st.to_string())
print(f'\nSP share of revenue: {st.loc["SP","revenue"]/st.revenue.sum():.1%}')
print(f'Top 3 states (SP,RJ,MG): {(st.head(3).revenue.sum()/st.revenue.sum()):.1%} of revenue')

# seller states
ss = items[['order_id','seller_id','price']].merge(sellers[['seller_id','seller_state']], on='seller_id')
ss = ss[ss['order_id'].isin(o['order_id'])]
sst = ss.groupby('seller_state').agg(orders=('order_id','nunique'), revenue=('price','sum')).sort_values('revenue',ascending=False)
print('\nSELLER STATE RANKING')
print(sst.head(10).to_string())
print(f'\nTop seller state {sst.index[0]}: {sst.revenue.iloc[0]/sst.revenue.sum():.1%} of seller revenue')

STATE REVENUE RANKING
                orders     revenue
customer_state                    
SP               41375  5202955.05
RJ               12762  1824092.67
MG               11544  1585308.03
RS                5432   750304.02
PR                4998   683083.76
SC                3612   520553.34
BA                3358   511349.99
DF                2125   302603.94
GO                2007   294591.95
ES                2025   275037.31
PE                1648   262788.03
CE                1327   227254.71
PA                 970   178947.81
MT                 903   156453.53
MA                 740   119648.22
MS                 709   116812.64
PB                 532   115268.08
PI                 493    86914.08
RN                 482    83034.98
AL                 411    80314.81
SE                 345    58920.85
TO                 279    49621.74
RO                 247    46140.64
AM                 147    22356.84
AC                  81    15982.95
AP                  68    13474.3


SELLER STATE RANKING
              orders     revenue
seller_state                    
SP             68641  8509511.46
PR              7512  1232096.99
MG              7735   977866.31
RJ              4227   820611.59
SC              3603   613591.65
RS              1962   373412.08
BA               550   277925.51
DF               808    94840.31
PE               403    91164.15
GO               451    64806.59

Top seller state SP: 64.4% of seller revenue


## 8. Delivery Performance — the operations bottleneck

In [10]:
d = o.copy()
d['delivery_days'] = (d['order_delivered_customer_date'] - d['order_purchase_timestamp']).dt.days
d['estimated_days'] = (d['order_estimated_delivery_date'] - d['order_purchase_timestamp']).dt.days
d['on_time'] = d['order_delivered_customer_date'] <= d['order_estimated_delivery_date']
d = d.dropna(subset=['delivery_days'])

print(f'Mean delivery days: {d.delivery_days.mean():.2f}  Median: {d.delivery_days.median():.1f}')
print(f'On-time delivery rate: {d.on_time.mean():.1%}')
print(f'Late orders: {d.on_time.value_counts()}  -> {d.on_time.value_counts().get(False,0):,} late ({d.on_time.mean():.1%} on-time)')

# by state
ocs = d[['order_id','delivery_days','on_time']].merge(oc[['order_id','customer_state']], on='order_id', how='left')
st_d = ocs.groupby('customer_state').agg(avg_days=('delivery_days','mean'), on_time=('on_time','mean'), n=('order_id','size'))
st_d = st_d.sort_values('avg_days', ascending=False)
print('\nSLOWEST 5 STATES (avg delivery days)')
print(st_d.head().to_string())
print('\nFASTEST 5 STATES')
print(st_d.tail().to_string())
print('\nStates with WORST on-time %')
print(st_d.sort_values('on_time').head().to_string())

Mean delivery days: 12.09  Median: 10.0
On-time delivery rate: 91.9%
Late orders: on_time
True     88644
False     7826
Name: count, dtype: int64  -> 7,826 late (91.9% on-time)



SLOWEST 5 STATES (avg delivery days)
                 avg_days   on_time     n
customer_state                           
RR              27.826087  0.891304    46
AP              27.753086  0.950617    81
AM              25.963190  0.957055   163
AL              23.992974  0.758782   427
PA              23.301708  0.875712  1054

FASTEST 5 STATES
                 avg_days   on_time      n
customer_state                            
SC              14.517208  0.903832   4097
DF              12.501486  0.925690   2355
MG              11.514091  0.945571  12916
PR              11.480793  0.952204   5649
SP               8.259663  0.942314  46441

States with WORST on-time %
                 avg_days   on_time     n
customer_state                           
AL              23.992974  0.758782   427
MA              21.203750  0.796250   800
SE              20.978667  0.837333   375
PI              18.931166  0.845124   523
CE              20.537167  0.847125  1426


## 9. Customer Satisfaction — review scores and the late-delivery effect

In [11]:
r = reviews[reviews['order_id'].isin(o['order_id'])]
print(f'Mean review score (delivered): {r.review_score.mean():.2f}')
print('\nScore distribution:')
print(r.review_score.value_counts().sort_index().to_string())
print(f'\n% negative (1-2): {(r.review_score<=2).mean():.1%}   % positive (4-5): {(r.review_score>=4).mean():.1%}')

# late effect
dr = d[['order_id','on_time']].merge(r[['order_id','review_score']], on='order_id', how='inner')
eff = dr.groupby('on_time').agg(avg=('review_score','mean'), n=('order_id','size'))
print('\nREVIEW SCORE: ON-TIME vs LATE')
print(eff.to_string())
print(f'\nSatisfaction gap: {eff.loc[True,"avg"]-eff.loc[False,"avg"]:.2f} points')

from scipy import stats
on = dr.loc[dr.on_time,'review_score']; lt = dr.loc[~dr.on_time,'review_score']
t, p = stats.ttest_ind(on, lt, equal_var=False)
print(f't-test: t={t:.1f}, p={p:.2e}  -> statistically significant: {p<0.001}')

Mean review score (delivered): 4.16

Score distribution:
review_score
1     9406
2     2941
3     7961
4    18987
5    57066

% negative (1-2): 12.8%   % positive (4-5): 78.9%



REVIEW SCORE: ON-TIME vs LATE
              avg      n
on_time                 
False    2.566494   7700
True     4.293718  88653

Satisfaction gap: 1.73 points


t-test: t=89.6, p=0.00e+00  -> statistically significant: True


### 9.1 Review score by delivery-time bucket

In [12]:
d2 = dr.copy()
d2['delivery_days'] = d2['order_id'].map(d.set_index('order_id')['delivery_days'])
bins = [-1,7,14,21,30,60,365]
labels = ['0-7d','8-14d','15-21d','22-30d','31-60d','60d+']
d2['bucket'] = pd.cut(d2['delivery_days'], bins=bins, labels=labels)
b = d2.groupby('bucket', observed=True).agg(avg_score=('review_score','mean'), n=('order_id','size'))
print(b.to_string())

        avg_score      n
bucket                  
0-7d     4.408307  33683
8-14d    4.288446  36395
15-21d   4.102334  15381
22-30d   3.494390   6863
31-60d   2.175672   3757
60d+     2.175182    274


### 9.2 Review score by state

In [13]:
drs = d2.merge(oc[['order_id','customer_state']], on='order_id', how='left')
st_r = drs.groupby('customer_state').agg(avg=('review_score','mean'), n=('order_id','size'))
print('LOWEST 5 REVIEW STATES')
print(st_r.sort_values('avg').head().to_string())
print('\nHIGHEST 5 REVIEW STATES')
print(st_r.sort_values('avg').tail().to_string())

LOWEST 5 REVIEW STATES
                     avg     n
customer_state                
MA              3.765957   799
AL              3.815421   428
PA              3.842459  1041
BA              3.860725  3669
CE              3.867275  1424

HIGHEST 5 REVIEW STATES
                     avg      n
customer_state                 
AC              4.131868     91
PR              4.144476   5648
TO              4.155340    309
SP              4.177189  46425
AP              4.262500     80


## 10. Seller Concentration — the long tail

In [14]:
si = items[items['order_id'].isin(o['order_id'])]
sagg = si.groupby('seller_id').agg(orders=('order_id','nunique'), revenue=('price','sum'), items=('order_item_id','size'))
print(f'Active sellers: {len(sagg):,}')
print(f'Median seller revenue: R$ {sagg.revenue.median():,.0f}')
print(f'Mean seller revenue: R$ {sagg.revenue.mean():,.0f}')
print(f'Top 1% sellers share: {sagg.revenue.sort_values(ascending=False).head(max(1,len(sagg)//100)).sum()/sagg.revenue.sum():.1%}')
print(f'Top 10 sellers share: {sagg.revenue.nlargest(10).sum()/sagg.revenue.sum():.1%}')
print(f'Bottom 50% sellers share: {sagg.revenue.sort_values(ascending=False).tail(len(sagg)//2).sum()/sagg.revenue.sum():.1%}')
print('\nGini-style look — revenue shares:')
top20 = sagg.revenue.nlargest(int(len(sagg)*0.2)).sum()/sagg.revenue.sum()
print(f'  Top 20% of sellers -> {top20:.1%} of revenue')

Active sellers: 2,970
Median seller revenue: R$ 846
Mean seller revenue: R$ 4,452


Top 1% sellers share: 25.5%
Top 10 sellers share: 13.3%
Bottom 50% sellers share: 3.3%

Gini-style look — revenue shares:
  Top 20% of sellers -> 82.3% of revenue


## 11. Correlations & Segments

In [15]:
# order-level panel
panel = o[['order_id','price','freight','m']].merge(r[['order_id','review_score']], on='order_id', how='left')
panel = panel.merge(oc[['order_id','customer_state']], on='order_id', how='left')
panel = panel.merge(d[['order_id','delivery_days','on_time']], on='order_id', how='left')

print('CORRELATION MATRIX (order level)')
print(panel[['price','freight','review_score','delivery_days']].corr().to_string())
print('''
Interpretation (business):
- delivery_days vs review_score = NEGATIVE -> the longer the wait, the worse the rating
- price vs review_score = small positive -> expensive orders slightly better rated
- freight vs price = strong positive -> heavier/costlier items carry higher freight''')

# AOV segments
panel['aov_seg'] = pd.cut(panel['price'], [0,50,100,200,400,1e9], labels=['<50','50-100','100-200','200-400','400+'])
seg = panel.groupby('aov_seg', observed=True).agg(orders=('order_id','size'), avg_score=('review_score','mean'), late_rate=('on_time', lambda x: (x == False).mean()))
print('\nSEGMENT: ORDER VALUE vs SCORE & LATENESS')
print(seg.to_string())

CORRELATION MATRIX (order level)


                  price   freight  review_score  delivery_days
price          1.000000  0.415505     -0.070697       0.050143
freight        0.415505  1.000000     -0.134947       0.125520
review_score  -0.070697 -0.134947      1.000000      -0.304013
delivery_days  0.050143  0.125520     -0.304013       1.000000

Interpretation (business):
- delivery_days vs review_score = NEGATIVE -> the longer the wait, the worse the rating
- price vs review_score = small positive -> expensive orders slightly better rated
- freight vs price = strong positive -> heavier/costlier items carry higher freight

SEGMENT: ORDER VALUE vs SCORE & LATENESS
         orders  avg_score  late_rate
aov_seg                              
<50       30635   4.193812   0.074588
50-100    30647   4.125990   0.078605
100-200   29841   4.052950   0.081331
200-400   13002   3.924804   0.083603
400+       6715   3.790544   0.081459


### 11.1 Repeat vs one-time customers — is repeat higher value?

In [16]:
# repeat customers: customer_unique_id ordering more than once
cust_ord = customers.merge(orders, on='customer_id')[['customer_unique_id','order_id','order_purchase_timestamp']]
cust_ord = cust_ord[cust_ord['order_id'].isin(o['order_id'])]
freq = cust_ord.groupby('customer_unique_id')['order_id'].nunique()
repeat_ids = freq[freq>1].index
print(f'Repeat customers: {len(repeat_ids):,} of {len(freq):,} unique buyers')

# revenue per customer
crev = cust_ord.merge(items[['order_id','price']], on='order_id', how='inner').groupby('customer_unique_id')['price'].sum()
one = crev[~crev.index.isin(repeat_ids)]; rep = crev[crev.index.isin(repeat_ids)]
print(f'Avg spend 1x customer:  R$ {one.mean():,.0f}  (n={len(one):,})')
print(f'Avg spend repeat cust:  R$ {rep.mean():,.0f}  (n={len(rep):,})')
print(f'Repeat customers drive: {rep.sum()/crev.sum():.1%} of total revenue from {len(repeat_ids)/len(freq):.1%} of customers')
print(f'\nLifetime value uplift per repeat customer: x{rep.mean()/one.mean():.2f}')

Repeat customers: 2,801 of 93,358 unique buyers


Avg spend 1x customer:  R$ 138  (n=90,557)
Avg spend repeat cust:  R$ 260  (n=2,801)
Repeat customers drive: 5.5% of total revenue from 3.0% of customers

Lifetime value uplift per repeat customer: x1.89


## 12. Statistical Deep-Dive

In [17]:
print('DELIVERY DAYS DISTRIBUTION')
print(d['delivery_days'].describe().to_string())
q = d['delivery_days'].quantile([0.25,0.5,0.75,0.9,0.95,0.99])
print('\nQuantiles:'); print(q.to_string())
print(f'\nOrders delivered within 7 days: {(d.delivery_days<=7).mean():.1%}')
print(f'Within 14 days: {(d.delivery_days<=14).mean():.1%}')
print(f'Within 30 days: {(d.delivery_days<=30).mean():.1%}')

print('\n\nORDER VALUE (price) DISTRIBUTION')
print(o['price'].describe().to_string())
pq = o['price'].quantile([0.5,0.9,0.95,0.99])
print('\nQuantiles:'); print(pq.to_string())
print(f'\nShare of orders <= R$137 (AOV): {(o.price<=aov).mean():.1%}')

DELIVERY DAYS DISTRIBUTION
count    96470.000000
mean        12.093604
std          9.551380
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000

Quantiles:
0.25     6.0
0.50    10.0
0.75    15.0
0.90    23.0
0.95    29.0
0.99    46.0

Orders delivered within 7 days: 34.9%
Within 14 days: 72.7%
Within 30 days: 95.7%


ORDER VALUE (price) DISTRIBUTION
count    96478.000000
mean       137.041586
std        209.045198
min          0.850000
25%         45.900000
50%         86.575000
75%        149.900000
max      13440.000000

Quantiles:
0.50     86.575
0.90    269.000
0.95    399.000
0.99    990.000

Share of orders <= R$137 (AOV): 70.8%


### 12.1 Seasonality & special dates (Black Friday, Christmas)

In [18]:
o['dm'] = o['order_purchase_timestamp'].dt.to_period('M')
o['dow'] = o['order_purchase_timestamp'].dt.dayofweek
o['dom'] = o['order_purchase_timestamp'].dt.day
bf = o[(o['dm']==pd.Period('2017-11',freq='M'))]
print('NOV-2017 (Black Friday month):')
print(f'  Orders: {len(bf):,} vs monthly avg {len(o)/24:,.0f}  ({(len(bf)/(len(o)/24)-1):+.0%} vs avg)')
print(f'  Revenue: R$ {bf.price.sum():,.0f}')
dec17 = o[o['dm']==pd.Period('2017-12',freq='M')]
print('\nDEC-2017 (Christmas):')
print(f'  Orders: {len(dec17):,}  Revenue: R$ {dec17.price.sum():,.0f}')
print('\nDay-of-week order mix:')
print(o['dow'].map({0:'Mon',1:'Tue',2:'Wed',3:'Thu',4:'Fri',5:'Sat',6:'Sun'}).value_counts().to_string())

NOV-2017 (Black Friday month):
  Orders: 7,289 vs monthly avg 4,020  (+81% vs avg)
  Revenue: R$ 987,765

DEC-2017 (Christmas):
  Orders: 5,513  Revenue: R$ 726,033

Day-of-week order mix:
dow
Mon    15701
Tue    15503
Wed    15076
Thu    14323
Fri    13685
Sun    11635
Sat    10555


### 12.2 Weekday vs weekend behaviour

In [19]:
o['is_weekend'] = o['dow']>=5
wk = o.groupby('is_weekend').agg(orders=('order_id','size'), aov=('price','mean'))
print(wk.to_string())

            orders         aov
is_weekend                    
False        74288  137.462315
True         22190  135.633063


## 13. Export EDA summary to file (feeds the canvas)

In [20]:
summary = {
 'gross_revenue_brl': float(rev),
 'delivered_orders': int(len(o)),
 'aov_brl': float(aov),
 'unique_customers': int(len(c_orders)),
 'repeat_rate_pct': float((c_orders>1).mean()*100),
 'avg_delivery_days': float(d.delivery_days.mean()),
 'median_delivery_days': float(d.delivery_days.median()),
 'on_time_rate_pct': float(d.on_time.mean()*100),
 'mean_review_score': float(r.review_score.mean()),
 'score_on_time': float(eff.loc[True,'avg']),
 'score_late': float(eff.loc[False,'avg']),
 'satisfaction_gap': float(eff.loc[True,'avg']-eff.loc[False,'avg']),
 'peak_month_orders': str(g.orders.idxmax()),
 'peak_revenue_brl': float(g.revenue.max()),
 'top_category': str(cat.index[0]),
 'top_category_rev_brl': float(cat.iloc[0].revenue),
 'repeat_customers_rev_share_pct': float(rep.sum()/crev.sum()*100),
 'aov_repeat_multiple': float(rep.mean()/one.mean()),
 'sp_rev_share_pct': float(st.loc['SP','revenue']/st.revenue.sum()*100),
 'top1pct_sellers_share_pct': float(sagg.revenue.sort_values(ascending=False).head(max(1,len(sagg)//100)).sum()/sagg.revenue.sum()*100),
}
import json
with open(OUT/'eda_summary.json','w',encoding='utf-8') as f: json.dump(summary,f,indent=2)
print('Saved', OUT/'eda_summary.json')
for k,v in summary.items(): print(f'{k}: {v}')

Saved 06_AI\Outputs\Generated_Insights\eda_summary.json
gross_revenue_brl: 13221498.11
delivered_orders: 96478
aov_brl: 137.0415857501192
unique_customers: 96096
repeat_rate_pct: 3.1187562437562435
avg_delivery_days: 12.093604229294082
median_delivery_days: 10.0
on_time_rate_pct: 91.88763346117965
mean_review_score: 4.155716524320005
score_on_time: 4.2937182046856845
score_late: 2.5664935064935066
satisfaction_gap: 1.727224698192178
peak_month_orders: 2017-11
peak_revenue_brl: 987765.37
top_category: health_beauty
top_category_rev_brl: 1258681.34
repeat_customers_rev_share_pct: 5.509275453808615
aov_repeat_multiple: 1.8850124612476995
sp_rev_share_pct: 38.280543287049234
top1pct_sellers_share_pct: 25.513173635358932


## Done

> **Next steps:** update `00_Context/PROJECT_CANVAS.md` §4 with these deep-EDA numbers, then proceed to Phase 3 KPI formalisation.